### Aggregation and Modelling

#### Importing Packages and Loading Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

project_root = Path.cwd().parent
processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "reports" / "figures"

df = pd.read_parquet(processed_dir / "london_crime_wards.parquet")

In [2]:
df.head()

,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,source_file,ward_code,ward_name,borough
0,e7b720d0e1302d2d06db7b28b29132eb194864d44d7921...,2024-01,City of London Police,-0.106220,51.518275,On or near B500,E01000916,Camden 027B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,E05013662,Holborn & Covent Garden,Camden
1,e60a5ac62a80e866453254474137c3206417422c62f0c0...,2024-01,City of London Police,-0.107682,51.517786,On or near B521,E01000917,Camden 027C,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,E05009305,Farringdon Without,City of London
2,986f618142ec52b7f254e4b0549da2f17ceeb0e130db6c...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,E05013662,Holborn & Covent Garden,Camden
3,05dc27a88748356f6d59b0bd1389710ebfb42b37e565af...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Theft from the person,Status update unavailable,2024-01-city-of-london-street.csv,E05013662,Holborn & Covent Garden,Camden
4,373d78e2ccec5d05a547cd4bee19045a9e050042a0e6e7...,2024-01,City of London Police,-0.112096,51.515942,On or near Nightclub,E01000914,Camden 028B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,E05013662,Holborn & Covent Garden,Camden


#### Types of Crime to Model

In [3]:
# Average incidents per ward-month andper crime type
n_wards = df["ward_code"].nunique()
n_months = df["month"].nunique()
ward_months = n_wards * n_months

In [4]:
density = (
    df["crime_type"].value_counts()
    .div(ward_months)
    .sort_values(ascending=False)
    .round(2)
)
density

crime_type
Violence and sexual offences    30.87
Anti-social behaviour           27.72
Other theft                     12.64
Theft from the person           10.96
Vehicle crime                   10.69
Shoplifting                     10.21
Public order                     6.70
Criminal damage and arson        6.54
Burglary                         5.98
Drugs                            5.46
Robbery                          3.84
Bicycle theft                    1.75
Other crime                      1.45
Possession of weapons            0.60
Name: count, dtype: float64

In [6]:
# Threshold chosen from the density distribution
THRESHOLD = 5.0
modelled_types = density[density >= THRESHOLD].index.tolist()
modelled_types

['Violence and sexual offences',
 'Anti-social behaviour',
 'Other theft',
 'Theft from the person',
 'Vehicle crime',
 'Shoplifting',
 'Public order',
 'Criminal damage and arson',
 'Burglary',
 'Drugs']

In [7]:
# Keeping only the modelled types
scoped = df[df["crime_type"].isin(modelled_types)]

scoped.head()

,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,source_file,ward_code,ward_name,borough
0,e7b720d0e1302d2d06db7b28b29132eb194864d44d7921...,2024-01,City of London Police,-0.106220,51.518275,On or near B500,E01000916,Camden 027B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,E05013662,Holborn & Covent Garden,Camden
1,e60a5ac62a80e866453254474137c3206417422c62f0c0...,2024-01,City of London Police,-0.107682,51.517786,On or near B521,E01000917,Camden 027C,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,E05009305,Farringdon Without,City of London
2,986f618142ec52b7f254e4b0549da2f17ceeb0e130db6c...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Other theft,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,E05013662,Holborn & Covent Garden,Camden
3,05dc27a88748356f6d59b0bd1389710ebfb42b37e565af...,2024-01,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Theft from the person,Status update unavailable,2024-01-city-of-london-street.csv,E05013662,Holborn & Covent Garden,Camden
4,373d78e2ccec5d05a547cd4bee19045a9e050042a0e6e7...,2024-01,City of London Police,-0.112096,51.515942,On or near Nightclub,E01000914,Camden 028B,Theft from the person,Investigation complete; no suspect identified,2024-01-city-of-london-street.csv,E05013662,Holborn & Covent Garden,Camden


In [8]:
scoped.shape

(2155410, 14)

In [9]:
# Aggregating to counts per ward, per month, per crime type
agg = (
    scoped.groupby(["ward_code", "ward_name", "borough", "month", "crime_type"])
    .size()
    .reset_index(name="crime_count")
)

agg.head()

,ward_code,ward_name,borough,month,crime_type,crime_count
0,E05009288,Aldersgate,City of London,2024-01,Anti-social behaviour,2
1,E05009288,Aldersgate,City of London,2024-01,Other theft,1
2,E05009288,Aldersgate,City of London,2024-01,Public order,1
3,E05009288,Aldersgate,City of London,2024-01,Theft from the person,1
4,E05009288,Aldersgate,City of London,2024-01,Violence and sexual offences,1


In [10]:
agg.shape

(158367, 6)

In [11]:
# Complete grid: every ward x every month x every modelled type
all_wards = agg[["ward_code", "ward_name", "borough"]].drop_duplicates()
all_months = sorted(agg["month"].unique())
all_types = sorted(agg["crime_type"].unique())

grid = pd.MultiIndex.from_product(
    [all_wards["ward_code"], all_months, all_types],
    names=["ward_code", "month", "crime_type"]
).to_frame(index=False)

grid.head()

,ward_code,month,crime_type
0,E05009288,2024-01,Anti-social behaviour
1,E05009288,2024-01,Burglary
2,E05009288,2024-01,Criminal damage and arson
3,E05009288,2024-01,Drugs
4,E05009288,2024-01,Other theft


In [12]:
grid.shape

(168720, 3)

In [13]:
# Attaching counts combinations with no crime
model_table = grid.merge(
    agg[["ward_code", "month", "crime_type", "crime_count"]],
    on=["ward_code", "month", "crime_type"], how="left"
)

model_table.head()

,ward_code,month,crime_type,crime_count
0,E05009288,2024-01,Anti-social behaviour,2.0
1,E05009288,2024-01,Burglary,NaN
2,E05009288,2024-01,Criminal damage and arson,NaN
3,E05009288,2024-01,Drugs,NaN
4,E05009288,2024-01,Other theft,1.0


In [14]:
model_table.shape

(168720, 4)

In [18]:
# Filling NaN values as 0
model_table["crime_count"] = model_table["crime_count"].fillna(0).astype(int)
model_table.head()

,ward_code,month,crime_type,crime_count
0,E05009288,2024-01,Anti-social behaviour,2
1,E05009288,2024-01,Burglary,0
2,E05009288,2024-01,Criminal damage and arson,0
3,E05009288,2024-01,Drugs,0
4,E05009288,2024-01,Other theft,1


In [19]:
# Adding back Columns
model_table = model_table.merge(all_wards, on="ward_code", how="left")
model_table.head()

,ward_code,month,crime_type,crime_count,ward_name,borough
0,E05009288,2024-01,Anti-social behaviour,2,Aldersgate,City of London
1,E05009288,2024-01,Burglary,0,Aldersgate,City of London
2,E05009288,2024-01,Criminal damage and arson,0,Aldersgate,City of London
3,E05009288,2024-01,Drugs,0,Aldersgate,City of London
4,E05009288,2024-01,Other theft,1,Aldersgate,City of London


In [20]:
model_table = model_table.sort_values(
    ["ward_code", "crime_type", "month"]
).reset_index(drop=True)
model_table

,ward_code,month,crime_type,crime_count,ward_name,borough
0,E05009288,2024-01,Anti-social behaviour,2,Aldersgate,City of London
1,E05009288,2024-02,Anti-social behaviour,2,Aldersgate,City of London
2,E05009288,2024-03,Anti-social behaviour,2,Aldersgate,City of London
3,E05009288,2024-04,Anti-social behaviour,0,Aldersgate,City of London
4,E05009288,2024-05,Anti-social behaviour,0,Aldersgate,City of London
...,...,...,...,...,...,...
168715,E05014119,2025-08,Violence and sexual offences,14,West Dulwich,Lambeth
168716,E05014119,2025-09,Violence and sexual offences,17,West Dulwich,Lambeth
168717,E05014119,2025-10,Violence and sexual offences,15,West Dulwich,Lambeth
168718,E05014119,2025-11,Violence and sexual offences,24,West Dulwich,Lambeth


In [22]:
# Saving
out_path = processed_dir / "model_table.parquet"
model_table.to_parquet(out_path, index=False)

### Summary

The previous notebooks produced 2.28 million individual crimes, each tagged with a ward. But our model doesn't predict individual crimes, it predicts how many crimes happen in a given ward, in a given month, for a given crime type. This notebook turns the crime list into that prediction target.

Not all 14 crime types are worth forecasting. Some occur with so low frequency that at ward level, almost every month is zero with the occasional unpredictable spike and it would only distort the model's error metrics. We used density which is the average number of incidents per ward per month and set a threshold of 5. Ten crime types sit above it, from Violence and sexual offenses (31 per ward-month) down to Drugs (5.5). The four excluded types (Robbery, Bicycle theft, Other crime, Possession of weapons) fall below our set threshold.

With the ten types selected, the individual crimesvwere grouped into counts per (ward × month × crime type). This collapsed 2.28
million crime records into roughly 158,000 count rows.

A plain grouping has a problem. If a ward had no burglaries (0) in a given month, that combination simply doesn't exist as a row and the zero is invisible. But zero crimes is real, predictable information. A forecasting model that never sees zeros never learns what a quiet zero crime ward-month looks like, and will systematically over predict. To fix this, the complete grid of every ward × every month × every modelled type was built and filled every missing NaN values with an explicit zero. The table was sorted chronologically within each ward and crime type.